## Great, now that we discussed a little let's continue

Given that the current approach utilized by the authors lacks reproducibility, we will explore an alternative method by leveraging nf-core pipelines for data analysis.

Please explain, how we will achieve reproducibility for the course  with this approach.


We can improve reproducibility by using standardized nf-core pipelines. We need to specify the pipeline version, parameters and reference files, this ensures that the same analysis steps can be repeated.
Also we use Docker containers to run the pipeline with defined software versions and dependencies.

You have successfully downloaded 2 of the fastq files we will use in our study.

What is the next step if we want to first have a count table and check the quality of our fastq files? What is the pipeline called to do so?

The next step would be to use the nf-core rnaseq pipeline. This pipeline processes raw RNA-seq FASTQ files, performs quality control, aligns or quantifies sequencing reads, and generates a count table.

Analyze the 2 files using an nf-core pipeline.

What does this pipeline do?

Which are the main tools that will be used in the pipeline?

The pipeline performs quality control of the raw reads (using FastQC), it checks the integrity and format of the FASTQ files (using fq-lint). Then it trimms adapters and bases with poor quality with Trim Galore/ Cutadapt (when enabeled). 
The it performs an alignment of the RNA-seq reads to a reference genome using HISAT2 and it estimates the transcript abundance with Salmon. Finally the pipeline produces a MultiQC report which combines all quaity-control results.

As all other nf-core pipelines, the chosen pipeline takes in a samplesheet as input.

Use Python and pandas to create the samplesheet for your 2 samples. Feel free to make use of the table you created earlier today.

Choose your sample names wisely, they must be the connection of the results to the metadata. If you can't find the sample in the metadata later, the analysis was useless.

In [ ]:
# post here the command you used to run nf-core/rnaseq
nextflow run nf-core/rnaseq     -r 3.26.0     -profile docker     -c fetchngs_resources.config     
--input day_02/rnaseq_samplesheet.csv     
--outdir day_02/rnaseq_results     
--fasta day_02/reference/Mus_musculus.GRCm39.dna.chromosome.19.fa.gz     
--gtf day_02/reference/mouse_chr19.gtf    
--aligner hisat2     
--pseudo_aligner salmon  
-resume

Explain all the parameters you set and why you set them in this way.

nextflow run nf-core/rnaseq     
-r 3.26.0  (defines the version of the pipeline)  

-profile docker (Runs pipeline tools inside Docker containers.)

-c fetchngs_resources.config (load my config file)

--input day_02/rnaseq_samplesheet.csv (specify input)

--outdir day_02/rnaseq_results (specifiy output)

--fasta day_02/reference/Mus_musculus.GRCm39.dna.chromosome.19.fa.gz  (mouse reference genome sequence, only used chromosome 19 here to reduce computational time and memory requirements)

--gtf day_02/reference/mouse_chr19.gtf  (gene and transcript annotations for mouse chromosome 19)

--aligner hisat2 (used this aligner because it is more suitable for my memory resources)

--pseudo_aligner salmon  (hisat does not produce a count table so I included this to produce one)

Flags inside the config file:

igenomes_ignore = true (prevents automatic selection of iGenomes reference files)

min_mapped_reads = 0 (only using chromosome 19 will leave many reads unmapped, this avoids that samples get excluded)

## Browsing the results

How did the pipeline perform?

Pipeline completed successfully -

Completed at: 29-Sep-2026 21:56:56

Duration    : 4h 54m 36s

CPU hours   : 15.9 (13.8% cached)

Succeeded   : 88

Cached      : 6

For a more detailed overview about the performance we can take a look in the results folder under pipeline_info. Looking at the execution report, Salmon and HISAT2 appear to have the highest CPU usage and some of the longest execution times, while FastQC and Picard show particularly high memory consumption.

Explain the quality control steps. Are you happy with the quality and why. If not, why not.
Please give additional information on : 

Note: to answer this I took a look at the MultiQC report of all samples provided by our tutors and not just my run with the smallest two samples.

- ribosomal rRNA: High rRNA content is concerning since we want to measure mRNA and not rRNA, typically rRNA should get removed during library preparation so no sequencing effort is spent on rRNA. For information about the rRNA content we look under biotype counts -> double click on legend to see which sample has how many rRNA reads. 
Especialliy samples SRX19144481, SRX19144484 and SRX19144489 have a higher rRNA content (the more rRNA the worse).

- Duplication: The duplication analysis reveals considerable variation between samples, with duplication rates ranging from 23.1% to 88.0%. Some samples, particularly SRX19144481 and SRX19144489, show very high duplication rates. High duplication rates can indicate low library complexity (low number of unique reads, less genes covered), but high duplication rates are not necessarily problematic because highly expressed genes could also explain the many duplicates.

- GC content: Is expected to follow a normal distribution. Strong deviations or large GC content could be due to contamination which can lead to biases in the gene expression estimates.

Quality control steps the MultiQC report includes information about:
- Strandedness checks (here we check if the strandedness declared in the samplesheet matches the inferred strandedness in Salmon)
- FastQC of raw reads (Checks sequencing quality, GC content, adapter content (here marked red (bad) for all samples), duplicated sequences and more)
- trimming with TrimmGalore (trimming of adapters ad low-quality bases)
- FastQC with trimmed reads (adapter content for all samples is okey here)
- DupRadar (quality control of duplication rate (From Report: "Highly expressed genes can be expected to have a lot of duplicate reads, but high numbers of duplicates at low read counts can indicate low library complexity with technical duplication."))
- Picard (mark duplicate reads)
- QualiMap (gives us information about the genomic origin of the reads and gene coverage profile)
- RSeQC (RNA-seq-specific quality metrics: informs about distribution of mapped reads over genome features, inner distance/ insert size etc.)
- Samtools (provides alignment statistics)
- STAR (aligns reads to refernce genome and provides a mapping statistic)
- Sample relationships (based on final count matrix, helps us to understand how similar our samples are and to spot potential outliers (481, 484, 489 and 492 again are suspicious))
- Biotype Counts (distribution of reads across different RNA biotype)

I am not completly satisified with the quality but it seems to be okey if we exlude some samples (see below). Especially in the FastQC section, many samples show warnings or failures for per-sequence GC content and sequence duplication levels. Together with the high rRNA content observed in some samples, these results indicate potential quality issues that should be investigated further.

What are the possible steps that could lead to poorer results?

Poorer results could be caused by RNA degradation during extraction, inefficient rRNA depletion during library preparation (causes high rRNA content), too much PCR amplification (leads to high duplication rates), or just sequencing errors.

Would you exclude any samples? If yes, which and why?

I would explude the samples SRX19144481,SRX19144484 and SRX19144489 because their strandedness check fails, their per sequence GC differes from the expected normal distribution, high duplication levels, a high rRNA content and seem to be unsimilar to the other samples in the Sample relationship plots.
There are more reads with a abnormal GC content we could consider to exclude (for example 492, 491 and 485 and more).
In general we look for reads with high GC content, little alignment and high rRNA content as potential candidates.

What would you now do to continue the experiment? What are the scientists trying to figure out? Which packages on R or python would you use?

Next, we would perform differential gene expression analysis to identify genes that are significantly upregulated or downregulated between the experimental groups (How does chronic pain and oxycodone withdrawal affect gene expression?). Before this, we would exclude the suspicious samples identified in the MultiQC report.
Then for the differential expression analysis, we could use DESeq2 or edgeR in R, or PyDESeq2 in Python. After that we could perform a pathway enrichment analysis to investigate the biological functions associated with the differentially expressed genes, like the authors of the paper did.

when here: clean up after run! whole genome ref are downloaded......delete work/ when done, clean up docker delete unused images